# 3.5 Practical Multi-Agent Implementations

**Week 4 — Agentic AI & Multi-Agent Systems**

## Learning objectives
- Implement a CrewAI-style **customer service** MAS (Specialist + Quality Analyst)
- Implement a CrewAI-style **culinary/medical safety** MAS (Chef + Nutritionist)
- Add basic **agent observability** ("Teenager Framework": steps, inputs/outputs, cost, latency)
- Implement **human-in-the-loop** approval gating
- Track cost across an agent run

> Bridge to the capstones: the human-in-the-loop pattern here is a nice-to-have for **SupportPilot**
> (confidence-threshold escalation) but a **hard requirement** for **CareRoute** (mandatory
> category-based escalation for patient-safety reasons).


## 1. CrewAI Customer Service MAS

**Roles:** a *Customer Care Specialist* researches documentation and proposes a response; a
*Support Quality Analyst* reviews that response for accuracy before it goes out.

```python
# Real CrewAI sketch:
specialist = Agent(role="Customer Care Specialist",
                    goal="Resolve customer queries using company documentation",
                    backstory="An experienced support agent who always cites policy.")
analyst = Agent(role="Support Quality Analyst",
                goal="Catch inaccurate or non-compliant responses before they are sent",
                backstory="A meticulous reviewer focused on accuracy and tone.")
crew = Crew(agents=[specialist, analyst], tasks=[...])
```


In [ ]:
from dataclasses import dataclass, field
from typing import Callable, Dict, Any, List
import time, random

POLICY_KB = {
    "return_window": "Standard items can be returned within 30 days of delivery.",
    "personalised_items": "Personalised items are not eligible for return.",
}

def customer_care_specialist(query: str) -> Dict[str, Any]:
    if "personalised" in query.lower() or "custom" in query.lower():
        answer = POLICY_KB["personalised_items"]
    else:
        answer = POLICY_KB["return_window"]
    return {"query": query, "proposed_answer": answer}

def support_quality_analyst(case: Dict[str, Any]) -> Dict[str, Any]:
    # A real analyst agent would use an LLM call to check tone/accuracy against retrieved policy.
    accurate = case["proposed_answer"] in POLICY_KB.values()
    return {**case, "approved": accurate, "reviewer_notes": "Matches policy KB." if accurate else "Could not verify against policy KB."}

case = customer_care_specialist("Can I return a personalised mug I ordered?")
reviewed = support_quality_analyst(case)
for k, v in reviewed.items():
    print(f"{k:>16}: {v}")


## 2. CrewAI Medical/Culinary MAS — Chef + Nutritionist

**Roles:** a *Chef* proposes a recipe from given ingredients; a *Nutritionist* evaluates it for safety
constraints (e.g. diabetic-safe). This pattern generalises to any "propose, then domain-expert
validate" pair — exactly the shape reused by CareRoute's triage-then-clinician-review flow.


In [ ]:
def chef_agent(ingredients: List[str]) -> Dict[str, Any]:
    recipe = f"Stir-fry of {', '.join(ingredients)} with a light honey glaze."
    return {"ingredients": ingredients, "recipe": recipe}

def nutritionist_agent(recipe_case: Dict[str, Any], dietary_constraint: str) -> Dict[str, Any]:
    high_sugar_terms = ["honey", "sugar", "syrup", "glaze"]
    risky = dietary_constraint == "diabetic" and any(t in recipe_case["recipe"].lower() for t in high_sugar_terms)
    verdict = "REJECTED: high-sugar glaze is not diabetic-safe. Suggest a lemon-herb glaze instead." if risky else "APPROVED"
    return {**recipe_case, "dietary_constraint": dietary_constraint, "verdict": verdict}

recipe_case = chef_agent(["chicken", "broccoli", "carrots"])
result = nutritionist_agent(recipe_case, dietary_constraint="diabetic")
for k, v in result.items():
    print(f"{k:>18}: {v}")


Notice the Nutritionist doesn't just approve/reject — it gives an **actionable alternative**. This
is a good general pattern for validator agents: a bare rejection forces another full round-trip; a
rejection **with a concrete fix** often lets the drafting agent self-correct in one extra step.


## 3. Agent Observability — the "Teenager Framework"

Named for the idea that agents, like teenagers, need to be asked *"where did you go, what did you do,
how much did it cost, and how long did it take?"* Track, for every step:

- **inputs / outputs** — what went in, what came out
- **cost** — tokens or $ spent on that step
- **latency** — how long the step took
- **step identity** — which agent, which action


In [ ]:
@dataclass
class StepLog:
    step_no: int
    agent: str
    input_data: Any
    output_data: Any
    tokens_used: int
    latency_ms: float

class ObservableRun:
    def __init__(self):
        self.logs: List[StepLog] = []

    def track(self, agent_name: str, func: Callable, *args, tokens_estimate: int = 50, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        latency_ms = (time.perf_counter() - start) * 1000
        self.logs.append(StepLog(len(self.logs) + 1, agent_name, args, result, tokens_estimate, latency_ms))
        return result

    def report(self):
        total_tokens = sum(l.tokens_used for l in self.logs)
        total_latency = sum(l.latency_ms for l in self.logs)
        print(f"{'Step':<5}{'Agent':<28}{'Tokens':<10}{'Latency(ms)':<12}")
        for l in self.logs:
            print(f"{l.step_no:<5}{l.agent:<28}{l.tokens_used:<10}{l.latency_ms:<12.2f}")
        print(f"\nTOTAL tokens: {total_tokens}  |  TOTAL latency: {total_latency:.2f} ms")
        # Illustrative cost: $0.003 per 1K tokens (blended input/output estimate)
        print(f"Estimated cost: ${total_tokens/1000 * 0.003:.5f}")

run = ObservableRun()
case = run.track("CustomerCareSpecialist", customer_care_specialist, "Can I return a personalised mug I ordered?", tokens_estimate=120)
reviewed = run.track("SupportQualityAnalyst", support_quality_analyst, case, tokens_estimate=80)
run.report()


This lightweight `ObservableRun` wrapper is the pattern behind production observability tools
(e.g. LangSmith, AgentOps, or a custom logging layer): wrap every agent call, capture cost/latency/
input/output, and you get a full audit trail for free — critical once agents run unattended.


## 4. Human-in-the-Loop Design

Not every agent decision should be allowed to execute unattended. Two common gating strategies:

- **Confidence-threshold gating** — escalate only when the model itself signals low confidence
  (used by SupportPilot).
- **Category-based gating** — escalate *always*, regardless of confidence, for certain categories
  (used by CareRoute, and mandatory in any patient-safety or high-risk domain).


In [ ]:
ALWAYS_ESCALATE_CATEGORIES = {"chest_pain", "breathing_difficulty", "self_harm_mention"}

def confidence_gate(confidence: float, threshold: float = 0.75) -> bool:
    """Returns True if a human must review, based on confidence alone."""
    return confidence < threshold

def category_gate(category: str) -> bool:
    """Returns True if a human must review, regardless of confidence. Cannot be bypassed."""
    return category in ALWAYS_ESCALATE_CATEGORIES

def route_decision(category: str, confidence: float) -> str:
    if category_gate(category):
        return "ESCALATE_TO_HUMAN (hard category rule — confidence is irrelevant)"
    if confidence_gate(confidence):
        return "ESCALATE_TO_HUMAN (low confidence)"
    return "AUTO_RESOLVE"

test_cases = [
    ("billing_question", 0.92),
    ("billing_question", 0.4),
    ("chest_pain", 0.99),   # even near-perfect confidence must still escalate
]
for category, confidence in test_cases:
    print(f"category={category:<20} confidence={confidence:<5} -> {route_decision(category, confidence)}")


Notice the third test case: **99% confidence does not matter** for a hard-gated category. This is
the exact design principle CareRoute's specification calls out explicitly — evaluators will test that
an urgent symptom category cannot be talked around by rephrasing it to look routine or by a
high-confidence model output. Category gates must be enforced as a hard rule in code, never as
"advice" inside a prompt that the model could choose to ignore.


## Key Takeaways

- CrewAI-style "propose, then domain-expert review" pairs (Specialist/Analyst, Chef/Nutritionist)
  generalise to almost any drafting + validation task.
- A validator agent that suggests a **fix**, not just a rejection, speeds up multi-agent convergence.
- The **Teenager Framework** — track inputs, outputs, tokens/cost, and latency per step — is the
  minimum observability bar for any agent you'd trust in production.
- Human-in-the-loop gating comes in two flavours: **confidence-based** (tunable) and **category-based**
  (a hard rule that must never be bypassable by rephrasing or by model confidence).

## Check your understanding
1. Why is a validator agent that returns "REJECTED, try X instead" generally better than one that just
   returns "REJECTED"?
2. Name the four things the Teenager Framework tracks for every agent step.
3. Why must category-based escalation be enforced in code rather than left to the model's judgement?

Next: **3.6 MCP Preview** — a conceptual look at the Model Context Protocol, previewing what Week 5
will build end-to-end.
